In [8]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score,classification_report

In [9]:
sns.get_dataset_names()

['anagrams',
 'anscombe',
 'attention',
 'brain_networks',
 'car_crashes',
 'diamonds',
 'dots',
 'dowjones',
 'exercise',
 'flights',
 'fmri',
 'geyser',
 'glue',
 'healthexp',
 'iris',
 'mpg',
 'penguins',
 'planets',
 'seaice',
 'taxis',
 'tips',
 'titanic']

In [52]:
df=sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [53]:

#The Titanic dataset contains the variable `alive`, which directly represents survival status.
#Because `survived` is our target, `alive` must NOT be used as a predictor. Otherwise, the model would be given information that directly reveals the target.
df=df.drop(["alive"],axis=1)  #check correlation between survied and alive also

In [12]:
df_number=df.select_dtypes(include="number")

In [13]:
df_number

,survived,pclass,age,sibsp,parch,fare
0,0,3,22.0,1,0,7.2500
1,1,1,38.0,1,0,71.2833
2,1,3,26.0,0,0,7.9250
3,1,1,35.0,1,0,53.1000
4,0,3,35.0,0,0,8.0500
...,...,...,...,...,...,...
886,0,2,27.0,0,0,13.0000
887,1,1,19.0,0,0,30.0000
888,0,3,NaN,1,2,23.4500
889,1,1,26.0,0,0,30.0000


In [14]:
df_number.isnull().sum()

survived      0
pclass        0
age         177
sibsp         0
parch         0
fare          0
dtype: int64

In [15]:
df_number['age']=df_number['age'].fillna(df_number['age'].median())
#or df_number=df_number.fillna(df_number.median())

In [16]:
df_number.isnull().sum()

survived    0
pclass      0
age         0
sibsp       0
parch       0
fare        0
dtype: int64

In [17]:
df_number.isnull().sum()

survived    0
pclass      0
age         0
sibsp       0
parch       0
fare        0
dtype: int64

In [18]:
df_string =df.select_dtypes(include=["string","object","category"])

In [19]:
df_string

,sex,embarked,class,who,deck,embark_town
0,male,S,Third,man,NaN,Southampton
1,female,C,First,woman,C,Cherbourg
2,female,S,Third,woman,NaN,Southampton
3,female,S,First,woman,C,Southampton
4,male,S,Third,man,NaN,Southampton
...,...,...,...,...,...,...
886,male,S,Second,man,NaN,Southampton
887,female,S,First,woman,B,Southampton
888,female,S,Third,woman,NaN,Southampton
889,male,C,First,man,C,Cherbourg


In [20]:
df_string.isnull().sum()

sex              0
embarked         2
class            0
who              0
deck           688
embark_town      2
dtype: int64

In [21]:
df_string=df_string.fillna(df_string.mode().iloc[0])

In [22]:
df_string

,sex,embarked,class,who,deck,embark_town
0,male,S,Third,man,C,Southampton
1,female,C,First,woman,C,Cherbourg
2,female,S,Third,woman,C,Southampton
3,female,S,First,woman,C,Southampton
4,male,S,Third,man,C,Southampton
...,...,...,...,...,...,...
886,male,S,Second,man,C,Southampton
887,female,S,First,woman,B,Southampton
888,female,S,Third,woman,C,Southampton
889,male,C,First,man,C,Cherbourg


In [23]:
oe=OrdinalEncoder()
df_str_number=pd.DataFrame(
    oe.fit_transform(df_string.astype(str)),
    columns=df_string.columns,
    index=df_string.index
)
df_str_number.head()

,sex,embarked,class,who,deck,embark_town
0,1.0,2.0,2.0,1.0,2.0,2.0
1,0.0,0.0,0.0,2.0,2.0,0.0
2,0.0,2.0,2.0,2.0,2.0,2.0
3,0.0,2.0,0.0,2.0,2.0,2.0
4,1.0,2.0,2.0,1.0,2.0,2.0


In [24]:
df_str_number.columns.to_list()

['sex', 'embarked', 'class', 'who', 'deck', 'embark_town']

In [25]:
df_str_number.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sex          891 non-null    float64
 1   embarked     891 non-null    float64
 2   class        891 non-null    float64
 3   who          891 non-null    float64
 4   deck         891 non-null    float64
 5   embark_town  891 non-null    float64
dtypes: float64(6)
memory usage: 41.9 KB


In [26]:
df_final=pd.concat([df_number,df_str_number],axis=1)

In [27]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   age          891 non-null    float64
 3   sibsp        891 non-null    int64  
 4   parch        891 non-null    int64  
 5   fare         891 non-null    float64
 6   sex          891 non-null    float64
 7   embarked     891 non-null    float64
 8   class        891 non-null    float64
 9   who          891 non-null    float64
 10  deck         891 non-null    float64
 11  embark_town  891 non-null    float64
dtypes: float64(8), int64(4)
memory usage: 83.7 KB


In [28]:
y=df_final["survived"]
X=df_final.drop(columns=["survived"])
print(y.shape)
print(X.shape)
X.columns

(891,)
(891, 11)


Index(['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex', 'embarked', 'class',
       'who', 'deck', 'embark_town'],
      dtype='str')

In [29]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.20)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(712, 11)
(179, 11)
(712,)
(179,)


In [30]:
model = LogisticRegression(max_iter=1000)

In [31]:
model.fit(X_train,y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [32]:
model.coef_

array([[-0.46149469, -0.02927841, -0.30090297, -0.12696231,  0.00299143,
        -2.63573198, -0.12122447, -0.46814729, -0.04520282,  0.28663327,
        -0.12122447]])

In [33]:
model.intercept_

array([3.56358449])

In [34]:
y_pred=model.predict(X_test)
print(y_pred[:10])
y_prob=model.predict_proba(X_test)[:,1]#class 1 probability
print(y_prob[:10])

[0 0 0 1 1 1 1 0 1 1]
[0.1123542  0.2147818  0.12879881 0.88664305 0.75747769 0.92921468
 0.66521147 0.08578865 0.73845365 0.9343244 ]


In [35]:
#confusion matrix
#accuracy,precision,recall,f1

cm = confusion_matrix(y_test,y_pred)
print("confusion matrix:",cm)
accuracy=accuracy_score(y_test,y_pred)
print("accuracy :",accuracy)
precision=precision_score(y_test,y_pred)
print("precision:",precision)
recall=recall_score(y_test,y_pred)
print("recall :",recall)
f1=f1_score(y_test,y_pred)
print("f1:",f1)

print(classification_report(y_test, y_pred))

confusion matrix: [[90 15]
 [18 56]]
accuracy : 0.8156424581005587
precision: 0.7887323943661971
recall : 0.7567567567567568
f1: 0.7724137931034483
              precision    recall  f1-score   support

           0       0.83      0.86      0.85       105
           1       0.79      0.76      0.77        74

    accuracy                           0.82       179
   macro avg       0.81      0.81      0.81       179
weighted avg       0.81      0.82      0.82       179



In [36]:
#cm = confusion_matrix(y_test,y_pred)
#                             pred0 pred1
#confusion_matrix cn= actal0   TN    FP
#                     actal1   FN    TP
#accracy=(TN+TP)/(TN+TP+FN+FP)
#Presision=TP/(TP+FP)
#Recall,sensitivity,TPR =TP/(TP+FN)
#R-Score=  (2*pression*recall)/(pression+recall)
tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print("tn:",tn)
print("fp:",fp)
print("fn:",fn)
print("tp:",tp)


tn: 90
fp: 15
fn: 18
tp: 56


In [37]:
#pd.DataFrame({"X index:":X_test.index,"y_test":y_test,"y_pred":y_pred,"y_prob":y_prob})
#X index:	y_test	y_pred	y_prob
#709     709	1	0	0.112354
#439	    439	0	0	0.214782
#840	    840	0	0	0.128799
result=pd.DataFrame({"X index:":X_test.index,"Actual":y_test,"Predicted":y_pred,"Probability":y_prob})
result=result.reset_index(drop=True)
result.head()

,X index:,Actual,Predicted,Probability
0,709,1,0,0.112354
1,439,0,0,0.214782
2,840,0,0,0.128799
3,720,1,1,0.886643
4,39,1,1,0.757478


In [38]:
#write into Excel

#1.raw data,
#2.final data,
#3.training data
#4ctesting data
#5 coeff
#6 intercepts
#7confusin matrix,
#8. model evaluation metrics (accuracy,precision,recall,f-score)
#9 TP
#10 FP,
#11 FN
#12 TN


In [64]:
import os
os.chdir(r"F:\DataScience")
excel_file = "Titanic_Logistic_Regression_Book.xlsx"

with pd.ExcelWriter(
    excel_file,
    engine="xlsxwriter"
) as writer:

#1. Raw Data
    df.to_excel(
    writer,
    sheet_name="RawData"
    )
#2. Final Data
    final_export=df_final.copy()
    final_export["survived_actual"]= y
    final_export["prediction"]=model.predict(X)
    final_export["probability"]=model.predict_proba(X)[:,1]
    final_export.to_excel(
    writer,
    sheet_name="Final_Data"
    )
    
#3. Training Data
    train_export=X_train.copy()
    train_export["survived_actual"]= y_train
    train_export["prediction"]=model.predict(X_train)
    train_export["probability"]=model.predict_proba(X_train)[:,1]
    train_export.to_excel(
    writer,
    sheet_name="Training_Data"
    )

#4. Testing Data
    test_export=X_test.copy()
    test_export["survived_actual"]= y_test
    test_export["prediction"]=model.predict(X_test)
    test_export["probability"]=model.predict_proba(X_test)[:,1]
    test_export.to_excel(
    writer,
    sheet_name="Testing_Data"
    )

#5. Coefficents
    coeff_export=pd.DataFrame({"Features":X.columns,
                               "Coefficients":model.coef_.ravel()})
    coeff_export.to_excel(
    writer,
    sheet_name="Coefficients",
    index=False
    )

#6. intercept
    intercept_export=pd.DataFrame({"intercept":model.intercept_})
    intercept_export.to_excel(
    writer,
    sheet_name="intercept",
    index=False
    )

#7 confusion_matrix
    confusion_matrix_export=pd.DataFrame(
    cm,
    index=["Actual 0","Actual 1"],
    columns=["prediction 0","Prediction 1"]
    )
    confusion_matrix_export.to_excel(
    writer,
    sheet_name="Confusion_matrix",
    )
    
#8 Model evaluation
    model_evalation_export=pd.DataFrame({
    "Metrics":["accuracy","precision","recall","F1 Score"],
    "Values":[accuracy,precision,recall,f1]
    })
    model_evalation_export.to_excel(
    writer,
    sheet_name="model_evaluation",
    index=True
    )


#We separate the test predictions into four groups:

#True Positive (TP): Actual = 1 and Predicted = 1
#False Positive (FP): Actual = 0 and Predicted = 1
#False Negative (FN): Actual = 1 and Predicted = 0
#True Negative (TN): Actual = 0 and Predicted = 0

#Genarate Actual Vs Predicted 
    actual_predicted=pd.DataFrame({"Actual":y_test,"Predicted":y_pred,"Probability_survied":y_prob})
    actual_predicted["Correct"] = actual_predicted["Actual"] == actual_predicted["Predicted"]

    true_negative_export=actual_predicted[(actual_predicted["Actual"]==0) & (actual_predicted["Predicted"]==0)]
    true_positive_export=actual_predicted[(actual_predicted["Actual"] ==1) & (actual_predicted["Predicted"]==1)]
    false_positive_export=actual_predicted[(actual_predicted["Actual"]==0) & (actual_predicted["Predicted"]==1)]    
    false_negative_export=actual_predicted[(actual_predicted["Actual"]==1) & (actual_predicted["Predicted"]==0)]

    print(actual_predicted.shape)

    print("true_positive_export:",true_positive_export.shape)
    print("true_negative_export",true_negative_export.shape)
    print("false_positive_export",false_positive_export.shape)
    print("false_negative_export",false_negative_export.shape)
#9  (TP)
    true_positive_export.to_excel(
    writer,
    sheet_name="TP",
    )
#10 (TN)
    true_negative_export.to_excel(
    writer,
    sheet_name="TN"
    )
#11 (TP)
    false_positive_export.to_excel(
    writer,
    sheet_name="FP"
    )
#12 (TP)
    false_negative_export.to_excel(
    writer,
    sheet_name="FN"
    )
    print("Excel workbook created:", excel_file) 

(179, 4)
true_positive_export: (56, 4)
true_negative_export (90, 4)
false_positive_export (15, 4)
false_negative_export (18, 4)
Excel workbook created: Titanic_Logistic_Regression_Book.xlsx
